# Resistor Fault Detection
### Name: Brian Pugh
### Topic: Unsupervised Learning – Clustering & PCA

This notebook explores unsupervised learning using K-Means clustering, hierarchical clustering, and PCA on a synthetic resistor dataset.

## Setup and Data Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
print('Imports OK')

In [ ]:
# Synthetic resistor dataset
n = 300
resistance = rng.normal(100, 10, n)       # Ohms
tolerance = rng.uniform(1, 5, n)          # percent
temperature = rng.normal(25, 6, n)        # Celsius

# Inject a small faulty pocket: overheated and out-of-spec components
fault_idx = rng.choice(n, size=40, replace=False)
resistance[fault_idx] += rng.normal(20, 5, size=fault_idx.size)
temperature[fault_idx] += rng.normal(20, 5, size=fault_idx.size)

df = pd.DataFrame({
    'Resistance_Ohm': resistance,
    'Tolerance_pct': tolerance,
    'Temp_C': temperature
})

display(df.head())

X = df.values
Xs = StandardScaler().fit_transform(X)

print('Data shape:', X.shape)

## Task 1 – K-Means Clustering

In [ ]:
# K-Means with k=2 and k=3
for k in [2, 3]:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(Xs)
    df[f'k{k}_label'] = labels

    print(f'\nK={k} centroids in standardized space:')
    print(km.cluster_centers_)

    plt.figure(figsize=(6, 4))
    plt.scatter(Xs[:, 0], Xs[:, 1], c=labels, s=30)
    plt.xlabel('Standardized Resistance')
    plt.ylabel('Standardized Tolerance')
    plt.title(f'K-Means Clustering with k={k}')
    plt.grid(True)
    plt.show()

### K-Means Interpretation

The k=2 model is useful for separating the dataset into a likely normal group and a likely faulty group. The injected faulty samples have higher resistance drift and higher temperature, so they tend to separate from the normal operating points. The k=3 model gives a more detailed grouping, which may split the normal operating region into subgroups or isolate a more extreme faulty region.

## Task 2 – Hierarchical Clustering

In [ ]:
# Hierarchical clustering using Ward linkage
Z = linkage(Xs, method='ward')

plt.figure(figsize=(9, 4))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=45)
plt.title('Hierarchical Clustering Dendrogram (Truncated)')
plt.xlabel('Merged Cluster Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

### Hierarchical Clustering Interpretation

The dendrogram shows how samples merge into larger groups as the distance threshold increases. Larger jumps in linkage distance suggest stronger separation between groups. This supports the idea that the dataset contains a normal operating region and a smaller faulty/anomalous region.

## Task 3 – PCA Dimensionality Reduction

In [ ]:
# PCA projection to 2D
pca = PCA(n_components=2, random_state=0)
Xp = pca.fit_transform(Xs)

print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Total explained variance in 2D:', pca.explained_variance_ratio_.sum())

plt.figure(figsize=(6, 4))
plt.scatter(Xp[:, 0], Xp[:, 1], c=df['k2_label'].values, s=30)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA 2D Projection with K-Means k=2 Labels')
plt.grid(True)
plt.show()

### PCA Interpretation

PCA reduces the original three-dimensional dataset into two principal components. PC1 usually captures the largest direction of variation, which is strongly influenced by resistance and temperature drift in this dataset. PC2 captures additional variation, such as tolerance differences. The PCA plot helps visualize whether the K-Means clusters separate clearly in a lower-dimensional feature space.

## Task 4 – Comparison and Reflection

Unsupervised learning is different from supervised learning because it does not require labeled examples. In earlier supervised assignments, the model learned from known input-output examples. In this assignment, K-Means, hierarchical clustering, and PCA discover structure directly from the resistor dataset.

For circuit fault detection, clustering can separate normal resistor behavior from abnormal behavior caused by overheating or resistance drift. A cluster with unusually high temperature and resistance values may indicate components that are moving out of specification. This is useful because real engineering systems may not always have labeled failure data available.

For sensor grouping, clustering can identify sensors or circuit components that behave similarly under operating conditions. This can help engineers group devices by tolerance, temperature behavior, or reliability characteristics. PCA can also reduce the number of features needed for monitoring, making systems easier to visualize and potentially faster to process.

Overall, clustering and PCA can support anomaly detection, signal compression, quality control, and predictive maintenance. These methods complement supervised learning by revealing hidden patterns before labels are available.